In [ ]:
from math import *
from build123d import *
from cad_utils import *

set_port(3939)
set_viewer_config(port=3939)

In [ ]:
diameter = 8
height = 8

plunger_size = 2.8
plunger_tolerance = 0.2
plunger_ridges = (1.0, 0.5, 0.2)
plunger_depth = 4


In [ ]:
reset_show()


def pyramid(x, y, height, mode: Mode = Mode.ADD):
  dx, dy, dz = x, height, y
  Wedge(
    dx,
    dy,
    dz,
    dx / 2,
    dz / 2,
    dx / 2,
    dz / 2,
    rotation=(90, 0, 0),
    align=(Align.CENTER, Align.MIN, Align.CENTER),
    mode=mode,
  )


with BuildPart() as power_switch_cap:
  ps = plunger_size + plunger_tolerance

  with BuildSketch() as s:
    Circle(diameter / 2)
  extrude(amount=height)

  with BuildSketch() as s:
    Rectangle(ps, ps)
  extrude(amount=plunger_depth, mode=Mode.SUBTRACT)

  with Locations((0, 0, plunger_depth)):
    pyramid(ps, ps, ps / 2, mode=Mode.SUBTRACT)

  with BuildPart(mode=Mode.PRIVATE) as ridge:
    with BuildSketch(Plane.YZ) as s:
      Triangle(a=plunger_ridges[1], b=plunger_ridges[2], C=90, rotation=90)
    extrude(amount=ps, mode=Mode.ADD)
  x = (ps) / 2
  add(ridge.part.moved(Location((-x, x, plunger_ridges[0]))))
  add(ridge.part.moved(Location((x, -x, plunger_ridges[0]), 180)))

  chamfer(edges().filter_by(GeomType.LINE).group_by(Axis.Z)[0], ps / 10)
  fillet(edges().sort_by(Axis.Z).last, diameter / 10)

power_switch_cap = power_switch_cap.part
power_switch_cap.label = "power_switch_cap"
power_switch_cap.color = Color(0.5, 0.1, 0.1)

show_object(power_switch_cap)

In [ ]:
export(power_switch_cap)